# Kumo Relational (NVIDIA) on the IBM AML data – single notebook

**Kumo Relational** is NVIDIA's pretrained *relational foundation model*, based on *KumoRFM-2: Scaling Foundation Models for Relational Learning* (2026). It predicts by **in-context learning**: you give it labeled example rows together with their neighbourhoods in a relational database, and it predicts labels for new rows in the same forward pass. There is **no training step**. This notebook runs the open weights ([`nvidia/Kumo-Relational`](https://huggingface.co/nvidia/Kumo-Relational)) locally with NVIDIA's [`structured-data-models`](https://github.com/NVIDIA/structured-data-models) library (`sdm`), on CPU.

Task, split and metric are the same as in `multignn_pyg.ipynb`, so the two can be compared directly: classify each transaction as laundering or not, temporal 60/20/20 split, minority-class F1.

| Step | What happens here |
|---|---|
| Relational database | `transactions`, `accounts`, `entities`, `banks`, built from the raw Kaggle files (`full_data.csv` is not needed) |
| Sampler | Kumo's built-in temporal neighbour sampler: transaction → its two accounts → their owner, bank and other transactions |
| Labeled examples | 10,000 training transactions including every training laundering case (the model's "training set", given as input) |
| Preprocessing | Kumo's default recipe, minus its calendar features (section 5 explains why) |
| Prediction | zero-shot: the pretrained weights are not updated (optional fine-tuning in section 10) |
| Threshold | chosen on validation to maximise F1, then applied to test |

Install once into the notebook's environment (match `torch-2.10.0` to your torch version):

```
pip install structured-data-models==0.2.1 pyg_lib -f https://data.pyg.org/whl/torch-2.10.0+cpu.html
```

The first run downloads the model weights (~120 MB) from Hugging Face.

In [1]:
import time, itertools, copy
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve
import sdm
import sdm.processing as sp
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kw): return it

# ---------------- paths ----------------
data_path = "/Users/joaopanicali/CS 224W/project/Data"   # REPLACE WITH YOUR LOCAL PATH (folder with the raw Kaggle files)

# ---------------- how much data ----------------
# Quick test: only the first MAX_EDGES transactions (chronologically). None = whole file (~5M transactions).
MAX_EDGES = None

# ---------------- what the sampler may see (section 4) ----------------
# "strict": a transaction only sees transactions up to its own timestamp (real-time detection).
# "window": it sees everything up to the end of its split, like Multi-GIN's per-split graphs.
TIME_MODE = "strict"
NUM_NEIGHBORS = [16, 16]   # neighbours sampled per hop, one entry per hop

# ---------------- labeled examples (section 5) ----------------
CONTEXT_SIZE = 10_000      # labeled training transactions given to the model, per ensemble member
N_POS = None               # how many of them are laundering; None = every training laundering case (2,530)
NUM_ESTIMATORS = 3         # ensemble members, each with its own draw of examples (scoring cost x NUM_ESTIMATORS)
# Calendar fields Kumo adds to every timestamp. Its default is ("minute", "hour", "weekday", "day_of_month", "month");
# on this 18-day dataset they hurt (section 5), so none are added.
CALENDAR_FIELDS = ()

# ---------------- evaluation (section 6) ----------------
BATCH_SIZE = 2000          # transactions scored per model call
# Scoring runs at ~120-250 transactions/s per ensemble member on CPU. Every laundering transaction is scored, but
# only this share of the legitimate ones; metrics re-weight them so they estimate the full-split values.
# 1.0 = exact (~1-2 h per split and ensemble member). The defaults take ~1 h with NUM_ESTIMATORS = 3.
NEG_RATE = dict(val=0.05, test=0.1)
SEED = 1

# Apple's MPS backend lacks an op the model needs, and with CPU fallback it is no faster than the CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| torch", torch.__version__, "| sdm", sdm.__version__)

device: cpu | torch 2.10.0 | sdm 0.2.1


## 1. Load the raw tables
* The transactions file has two columns named `Account` (sender, receiver); pandas reads the second one as `Account.1`.
* Bank IDs are read as integers, so a bank written `010` in one file matches `10` in the other.
* Transactions are sorted by time (stable sort), and `txn_id` is the position in that order.
* `HI-Small_Patterns.txt` is parsed only to tag illicit test transactions with their laundering pattern for the per-pattern recall table (same parsing as `data_processing.ipynb`). It is never given to the model.

In [2]:
txn = pd.read_csv(f"{data_path}/HI-Small_Trans.csv", dtype={"Account": str, "Account.1": str})
txn.columns = ["timestamp", "from_bank", "from_account", "to_bank", "to_account", "amount_received",
               "receiving_currency", "amount_paid", "payment_currency", "payment_format", "is_laundering"]
txn["timestamp"] = pd.to_datetime(txn["timestamp"], format="%Y/%m/%d %H:%M")
txn = txn.sort_values("timestamp", kind="stable").reset_index(drop=True)
if MAX_EDGES:
    txn = txn.iloc[:MAX_EDGES].copy()
txn["txn_id"] = np.arange(len(txn), dtype=np.int64)

acct = pd.read_csv(f"{data_path}/HI-Small_accounts.csv", dtype={"Account Number": str, "Entity ID": str})
acct.columns = ["bank_name", "bank_id", "account_number", "entity_id", "entity_name"]

# laundering pattern of each illicit transaction -- evaluation only
pattern_rows, laundering_type = [], None
with open(f"{data_path}/HI-Small_Patterns.txt") as f:
    for line in f:
        line = line.strip()
        if line.startswith("BEGIN LAUNDERING ATTEMPT"):
            # e.g. "BEGIN LAUNDERING ATTEMPT - FAN-OUT:  Max 16-degree Fan-Out" -> "Fan-Out"
            laundering_type = line.split(" - ", 1)[1].split(":")[0].strip().title()
        elif line and not line.startswith("END LAUNDERING ATTEMPT"):
            pattern_rows.append(line.split(",")[:10] + [laundering_type])
txn_key = ["timestamp", "from_bank", "from_account", "to_bank", "to_account", "amount_received",
           "receiving_currency", "amount_paid", "payment_currency", "payment_format"]
patterns = pd.DataFrame(pattern_rows, columns=txn_key + ["laundering_type"])
patterns["timestamp"] = pd.to_datetime(patterns["timestamp"], format="%Y/%m/%d %H:%M")
patterns[["from_bank", "to_bank"]] = patterns[["from_bank", "to_bank"]].astype(int)
patterns[["amount_received", "amount_paid"]] = patterns[["amount_received", "amount_paid"]].astype(float)
assert not patterns.duplicated(txn_key).any(), "a transaction appears in more than one laundering attempt"
n_rows = len(txn)
txn = txn.merge(patterns, on=txn_key, how="left")          # a left merge keeps the row order
assert len(txn) == n_rows and (txn.txn_id.to_numpy() == np.arange(n_rows)).all()

y = txn.is_laundering.to_numpy().astype(np.int64)
print(f"{len(txn):,} transactions | {len(acct):,} accounts | illicit: {y.sum():,} ({y.mean()*100:.3f}%), "
      f"{txn.laundering_type.notna().sum():,} of them in a laundering pattern")

5,078,345 transactions | 518,581 accounts | illicit: 5,177 (0.102%), 3,209 of them in a laundering pattern


## 2. Build the relational database
```
transactions ──from_account_id──▶ accounts ──entity_key──▶ entities
             ──to_account_id────▶          ──bank_id─────▶ banks
```
* **`transactions`**: one row per transaction: time, amounts (log-scaled, they are heavy-tailed), currencies, payment format. **The label is not a column here.** It lives only in the task table of section 4, so the model never reads the labels of neighbouring transactions.
* **`accounts`**: one row per (bank, account number), linking to its owner and its bank.
* **`entities`**: account owners, typed from their name (Corporation, Partnership, Sole Proprietorship, …). Many own several accounts, so this table links accounts across banks. Multi-GIN does not use this link.
* **`banks`**: country taken from the bank name (`"Germany Bank #4815"` → Germany; names without that pattern → Other).

Semantic types tell Kumo how to treat each column: `id` columns are keys, `numerical` and `categorical` columns are features, and the `datetime` column drives temporal sampling and relative-time features.

In [3]:
acct["account_id"] = np.arange(len(acct), dtype=np.int64)
acct_index = pd.Series(acct.account_id.to_numpy(),
                       index=pd.MultiIndex.from_arrays([acct.bank_id, acct.account_number]))
for side in ("from", "to"):
    ids = acct_index.reindex(pd.MultiIndex.from_arrays([txn[f"{side}_bank"], txn[f"{side}_account"]])).to_numpy()
    assert not pd.isna(ids).any(), f"some {side}-accounts are missing from the accounts file"
    txn[f"{side}_account_id"] = ids.astype(np.int64)
acct["entity_key"] = pd.factorize(acct.entity_id)[0].astype(np.int64)

entities = acct.drop_duplicates("entity_key")[["entity_key", "entity_name"]].reset_index(drop=True)
entities["entity_type"] = entities.entity_name.str.split("#").str[0].str.strip()
banks = acct.drop_duplicates("bank_id")[["bank_id", "bank_name"]].reset_index(drop=True)
banks["country"] = banks.bank_name.str.extract(r"^(.+?) Bank #\d+$")[0].fillna("Other")

trans_df = txn[["txn_id", "timestamp", "from_account_id", "to_account_id", "amount_received", "amount_paid",
                "receiving_currency", "payment_currency", "payment_format"]].copy()
trans_df[["amount_received", "amount_paid"]] = np.log1p(trans_df[["amount_received", "amount_paid"]])

data = sdm.RelationalData(
    tables={
        "transactions": sdm.TableTensor.from_pandas(trans_df, stypes={
            "txn_id": "id", "timestamp": "datetime", "from_account_id": "id", "to_account_id": "id",
            "amount_received": "numerical", "amount_paid": "numerical", "receiving_currency": "categorical",
            "payment_currency": "categorical", "payment_format": "categorical"}),
        "accounts": sdm.TableTensor.from_pandas(acct[["account_id", "entity_key", "bank_id"]], stypes={
            "account_id": "id", "entity_key": "id", "bank_id": "id"}),
        "entities": sdm.TableTensor.from_pandas(entities[["entity_key", "entity_type"]], stypes={
            "entity_key": "id", "entity_type": "categorical"}),
        "banks": sdm.TableTensor.from_pandas(banks[["bank_id", "country"]], stypes={
            "bank_id": "id", "country": "categorical"}),
    },
    relationships=[
        dict(left_table="transactions", left_column="from_account_id", right_table="accounts", right_column="account_id"),
        dict(left_table="transactions", left_column="to_account_id",   right_table="accounts", right_column="account_id"),
        dict(left_table="accounts",     left_column="entity_key",      right_table="entities", right_column="entity_key"),
        dict(left_table="accounts",     left_column="bank_id",         right_table="banks",    right_column="bank_id"),
    ],
)
for name, table in data.tables.items():
    cols = {str(s): list(c) for s, c in table.columns.items() if len(c)}
    print(f"{name:13s} {table.size(0):>10,} rows  {cols}")
n_acc = acct.groupby("entity_key").size()
print(f"\n{(n_acc > 1).sum():,} of {len(n_acc):,} entities own more than one account (max {n_acc.max():,})")

transactions   5,078,345 rows  {'id': ['txn_id', 'from_account_id', 'to_account_id'], 'datetime': ['timestamp'], 'numerical': ['amount_received', 'amount_paid'], 'categorical': ['receiving_currency', 'payment_currency', 'payment_format']}
accounts         518,581 rows  {'id': ['account_id', 'entity_key', 'bank_id']}
entities         166,207 rows  {'id': ['entity_key'], 'categorical': ['entity_type']}
banks             30,470 rows  {'id': ['bank_id'], 'categorical': ['country']}

63,518 of 166,207 entities own more than one account (max 7,820)


## 3. Temporal 60/20/20 split
Same search as in `multignn_pyg.ipynb`: two cut points on day boundaries, with proportions closest to 60/20/20. Because the cuts fall on day boundaries, the three sets hold exactly the same transactions as in that notebook (checked below on the full data).

In [4]:
def temporal_split(t_sec, unit, props=(0.6, 0.2, 0.2)):
    unit_idx = (t_sec // unit).astype(int)
    counts = np.bincount(unit_idx, minlength=unit_idx.max() + 1)
    total, best, best_err = counts.sum(), None, np.inf
    cs = np.r_[0, np.cumsum(counts)]
    for i, j in itertools.combinations(range(len(counts)), 2):
        tot = np.array([cs[i], cs[j] - cs[i], total - cs[j]]) / total
        err = np.max(np.abs(tot - props) / props)
        if err < best_err: best, best_err = (i, j), err
    i, j = best
    return int(cs[i]), int(cs[j])          # number of transactions in train / train+val

t_sec = (txn.timestamp - txn.timestamp.min()).dt.total_seconds().to_numpy()
unit = 86400 if t_sec.max() >= 10 * 86400 else 3600      # days normally; hours for short quick tests
n_tr, n_trval = temporal_split(t_sec, unit)
n_all = len(txn)
print(f"split unit: {'day' if unit == 86400 else 'hour'} | train {n_tr:,} | val {n_trval-n_tr:,} | test {n_all-n_trval:,}")
for name, sl in [("train", slice(0, n_tr)), ("val", slice(n_tr, n_trval)), ("test", slice(n_trval, n_all))]:
    print(f"  {name:5s} illicit ratio: {y[sl].mean()*100:.3f}%  ({y[sl].sum()} illicit)")
if MAX_EDGES is None:
    assert (n_tr, n_trval - n_tr, n_all - n_trval) == (3_248_921, 965_524, 863_900), "split differs from multignn_pyg.ipynb"

split unit: day | train 3,248,921 | val 965,524 | test 863,900
  train illicit ratio: 0.078%  (2530 illicit)
  val   illicit ratio: 0.107%  (1036 illicit)
  test  illicit ratio: 0.186%  (1611 illicit)


## 4. Configure Kumo's sampler
Kumo's sampler builds a separate small subgraph around every transaction, both the labeled examples and the ones being classified. With `NUM_NEIGHBORS = [16, 16]`:
* **hop 1**: the transaction's sender and receiver accounts;
* **hop 2**: each account's owner entity and bank, plus up to 16 transactions it sent and up to 16 it received.

A third hop would add the counterparties of those transactions, the owner's other accounts, and other accounts at the same bank. It did worse in an early validation check; the ablation in section 9 compares it.

The sampler only picks transactions whose timestamp is at or before each row's **seed time**, which `TIME_MODE` sets:
* **`"strict"`** (default): seed time = the transaction's own timestamp, and the most recent neighbours are taken. Nothing after the transaction is visible, as in real-time detection.
* **`"window"`**: seed time = the last timestamp of the row's split, with neighbours drawn uniformly. This mirrors `multignn_pyg.ipynb`, where each split has its own graph (train / train+val / everything) and neighbour sampling ignores time. A test transaction can then see *later* test transactions, e.g. the rest of a fan-out that is still in progress.

  Measured on this data, `"window"` is much worse than `"strict"`: validation AUPRC 0.22 ± 0.07 vs 0.39 ± 0.03 (3 runs each, 10,000 examples). Kumo describes every neighbour's time relative to the seed time, and moving the seed time to the end of the split shifts those times differently per split (train covers 6 days, validation 2, test 10).

Neither mode crosses into a later split: labeled training examples never see validation or test transactions.

The **task table** lists the transactions to sample around: key, seed time and label. Its label column gets a fixed category order `[0, 1]`. Otherwise the order follows whichever class appears first, and the model's probability columns can swap. Before this was fixed, some runs came out inverted (AUROC 0.08, i.e. the ranking was backwards).

In [5]:
split_of = np.repeat(np.array(["train", "val", "test"]), [n_tr, n_trval - n_tr, n_all - n_trval])
end_time = {"train": txn.timestamp.iloc[n_tr - 1], "val": txn.timestamp.iloc[n_trval - 1],
            "test": txn.timestamp.iloc[n_all - 1]}
timestamps = txn.timestamp.to_numpy()

def task_table(idx, time_mode=TIME_MODE):
    # the transactions to sample around: key, seed time, label
    seed_time = timestamps[idx] if time_mode == "strict" else pd.Series(split_of[idx]).map(end_time).to_numpy()
    df = pd.DataFrame({"txn_id": idx, "seed_time": seed_time,
                       "label": pd.Categorical(y[idx], categories=[0, 1])})
    return sdm.TableTensor.from_pandas(df, stypes={"txn_id": "id", "seed_time": "datetime", "label": "categorical"})

t0 = time.time()
sampler = data.sampler(time_columns={"transactions": "timestamp"})
print(f"sampler ready in {time.time()-t0:.1f}s")

def sample(task, time_mode=TIME_MODE, num_neighbors=NUM_NEIGHBORS):
    return sampler(task, task_link={"task_column": "txn_id", "table": "transactions", "table_column": "txn_id"},
                   num_neighbors=num_neighbors, task_time_column="seed_time",
                   temporal_strategy="last" if time_mode == "strict" else "uniform")

# sanity check: the subgraph around one fan-out laundering transaction from the test set
ex = int(np.flatnonzero((txn.laundering_type == "Fan-Out").to_numpy() & (split_of == "test"))[0])
for mode in ("strict", "window"):
    _, rel = sample(task_table(np.array([ex]), mode), mode)
    sizes = {name: int(t.size(0)) for name, t in rel.tables.items()}
    latest = pd.Timestamp(int(rel.tables["transactions"]["timestamp"].datetime.max()), unit="us")
    print(f"{mode:6s}: {sizes} | latest neighbour {latest} (transaction itself: {txn.timestamp.iloc[ex]})")

sampler ready in 1.0s
strict: {'transactions': 37, 'accounts': 2, 'entities': 2, 'banks': 2} | latest neighbour 2022-09-09 01:10:00 (transaction itself: 2022-09-09 01:10:00)
window: {'transactions': 53, 'accounts': 2, 'entities': 2, 'banks': 2} | latest neighbour 2022-09-11 09:42:00 (transaction itself: 2022-09-09 01:10:00)


## 5. Labeled examples (the in-context "training set")
Kumo learns the task from the labeled examples it is given. Only 0.08% of training transactions are laundering, so a random sample of 10,000 would contain about 8 of them. Instead, every training laundering case (2,530) is included, and the rest of the 10,000 are random legitimate training transactions, so 25% of the examples are laundering. Oversampling makes the predicted probabilities too high, so section 7 picks the decision threshold on validation instead of using 0.5.

The whole training set can't be the examples. The model compares every example with every other one, so cost grows quadratically, and when encoding each table it uses at most 20,000 labeled rows as reference. What mattered was including all laundering cases. Single-model runs, with validation and test scored on all laundering cases plus 1% of legitimate transactions:

| Examples | Laundering among them | val AUPRC | test AUPRC | test recall | Fit time |
|---|---|---|---|---|---|
| 4,000 (3 runs) | 800 | 0.19 ± 0.09 | 0.21 ± 0.11 | 0.23 ± 0.03 | 11 s |
| **10,000 (3 runs)** | **all 2,530** | **0.39 ± 0.03** | **0.42 ± 0.04** | **0.41 ± 0.02** | 35 s |
| 20,000 (2 runs) | all 2,530 | 0.38 ± 0.08 | 0.40 ± 0.08 | 0.29 ± 0.09 | 126 s |

`fit` encodes the examples and their subgraphs once and caches them. Scoring then only processes the new transactions.

Results also vary from run to run, even with the same examples: on each table, the model keeps a random 20,000 of the labeled rows as reference. So by default `NUM_ESTIMATORS = 3`: each ensemble member gets its own draw of legitimate examples, and their predictions are averaged.

**Preprocessing.** Before the model sees the tables, Kumo applies a preprocessing *recipe*. By default it adds calendar fields (minute, hour, weekday, day of month, month) to every timestamp in the sampled tables. That suits datasets spanning years. This one spans 18 days, so the day of month just says how late a transaction is: the labeled examples are from Sept 1–6, validation is Sept 7–8, and test is Sept 9–18. Every test transaction therefore falls outside the range of the examples. Test is also unusual: after Sept 10 there are only ~1,100 transactions, 60% of them laundering (40% of the test set's laundering cases). On one draw of examples, dropping the calendar fields raised validation AUPRC from 0.25 to 0.36 and test AUPRC from 0.21 to 0.40. The model still gets how long before each transaction its neighbouring transactions happened (relative time), which is what the laundering patterns depend on. `CALENDAR_FIELDS` adds some or all of the fields back.

In [6]:
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
tr_pos, tr_neg = np.flatnonzero(y[:n_tr] == 1), np.flatnonzero(y[:n_tr] == 0)

def draw_context(rng, size=CONTEXT_SIZE, n_pos=N_POS):
    n_pos = len(tr_pos) if n_pos is None else min(n_pos, len(tr_pos))
    return np.sort(np.r_[rng.choice(tr_pos, n_pos, replace=False), rng.choice(tr_neg, size - n_pos, replace=False)])

def make_recipe(calendar_fields=CALENDAR_FIELDS):
    # Kumo Relational's default recipe is TabICLv2's plus calendar fields on the related tables' timestamps
    recipe = sdm.models.TabICLv2.default_recipe()
    if calendar_fields:
        recipe = recipe.prepend_features(sp.TableDispatch(
            related=sp.StypeDispatch(datetime=sp.AddCalendarFields(tuple(calendar_fields)))))
    return recipe

def fit(model, draws, time_mode=TIME_MODE, num_neighbors=NUM_NEIGHBORS, calendar_fields=CALENDAR_FIELDS):
    # draws: one index array of labeled training transactions per ensemble member
    ctx = task_table(np.concatenate(draws), time_mode)
    if len(draws) > 1:
        ctx = ctx.unflatten(0, (len(draws), len(draws[0])))
    x, related = sample(ctx, time_mode, num_neighbors).to(device)
    model.fit(x=x.drop_columns("label"), y=x["label"], related_tables=related, recipe=make_recipe(calendar_fields))

model = sdm.models.KumoRelational(task="classification", device=device)
print(f"Kumo Relational: {sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters (pretrained)")
draws = [draw_context(rng) for _ in range(NUM_ESTIMATORS)]
t0 = time.time()
fit(model, draws)
print(f"labeled examples: {NUM_ESTIMATORS} x {len(draws[0]):,} training transactions "
      f"({int(y[draws[0]].sum()):,} laundering each) | fit in {time.time()-t0:.1f}s")

Kumo Relational: 29.9M parameters (pretrained)


labeled examples: 3 x 10,000 training transactions (2,530 laundering each) | fit in 124.8s


## 6. Score validation and test
`predict_proba` returns the model's laundering probability for each transaction, in batches.

Scoring all ~1.8M validation and test transactions would take about 2 hours per ensemble member on CPU. By default every laundering transaction is scored, plus a random `NEG_RATE` share of the legitimate ones. Each scored legitimate transaction then counts with weight `1 / NEG_RATE` in every metric, which makes precision, F1 and AUPRC unbiased estimates of their full-split values. Recall and the per-pattern recalls are exact. For the final numbers you can set `NEG_RATE` to 1.0.

In [7]:
def predict_proba(idx, time_mode=TIME_MODE, num_neighbors=NUM_NEIGHBORS, desc=None):
    probs = []
    for i in tqdm(range(0, len(idx), BATCH_SIZE), desc=desc, leave=False):
        q = task_table(idx[i:i + BATCH_SIZE], time_mode).drop_columns("label")
        if NUM_ESTIMATORS > 1:
            q = q.expand(NUM_ESTIMATORS, *q.size())
        out = model.predict(*sample(q, time_mode, num_neighbors).to(device))
        j = list(out.columns[sdm.Stype.numerical]).index("1")       # look the class up by name
        probs.append(out.numerical[..., j].float().cpu().numpy())
    return np.concatenate(probs)

def eval_set(lo, hi, neg_rate, rng):
    # all illicit transactions in [lo, hi) + a random share of the legitimate ones, with weights
    ids = np.arange(lo, hi)
    pos, neg = ids[y[ids] == 1], ids[y[ids] == 0]
    if neg_rate < 1:
        neg = rng.choice(neg, int(round(len(neg) * neg_rate)), replace=False)
    idx = np.sort(np.r_[pos, neg])
    return idx, np.where(y[idx] == 1, 1.0, 1.0 / neg_rate)

val_idx, val_w = eval_set(n_tr, n_trval, NEG_RATE["val"], rng)
te_idx, te_w = eval_set(n_trval, n_all, NEG_RATE["test"], rng)

t0 = time.time()
predict_proba(val_idx[:BATCH_SIZE])
rate = min(BATCH_SIZE, len(val_idx)) / (time.time() - t0)
print(f"{rate:.0f} transactions/s -> val: {len(val_idx):,} to score (~{len(val_idx)/rate/60:.0f} min), "
      f"test: {len(te_idx):,} (~{len(te_idx)/rate/60:.0f} min)")

  0%|          | 0/1 [00:00<?, ?it/s]

66 transactions/s -> val: 49,260 to score (~12 min), test: 87,840 (~22 min)


In [8]:
t0 = time.time()
val_p = predict_proba(val_idx, desc="val")
te_p = predict_proba(te_idx, desc="test")
print(f"scored {len(val_idx) + len(te_idx):,} transactions in {(time.time()-t0)/60:.1f} min")

val:   0%|          | 0/25 [00:00<?, ?it/s]

test:   0%|          | 0/44 [00:00<?, ?it/s]

scored 137,100 transactions in 43.6 min


## 7. Metrics
The decision threshold is the one that maximises F1 on validation. Test F1, precision and recall use that threshold. AUPRC and AUROC do not depend on it.

In [9]:
def best_threshold(y_true, p, w):
    prec, rec, thr = precision_recall_curve(y_true, p, sample_weight=w)
    f1 = 2 * prec * rec / np.clip(prec + rec, 1e-12, None)
    k = int(np.nanargmax(f1[:-1]))                 # the last curve point has no threshold
    return float(thr[k])

def scores(y_true, p, w, thr):
    pred = p >= thr
    tp, fp, fn = w[pred & (y_true == 1)].sum(), w[pred & (y_true == 0)].sum(), w[~pred & (y_true == 1)].sum()
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    both = y_true.min() != y_true.max()
    return dict(f1=2 * prec * rec / (prec + rec) if prec + rec else 0.0, prec=prec, rec=rec,
                auprc=average_precision_score(y_true, p, sample_weight=w) if both else np.nan,
                auroc=roc_auc_score(y_true, p, sample_weight=w) if both else np.nan)

def report(name, val_p, te_p):
    thr = best_threshold(y[val_idx], val_p, val_w)
    v, t = scores(y[val_idx], val_p, val_w, thr), scores(y[te_idx], te_p, te_w, thr)
    print(f"{name} | threshold {thr:.3f} (chosen on val)")
    for split, s in (("val", v), ("test", t)):
        print(f"  {split:4s} F1 {s['f1']:.4f} (P {s['prec']:.3f}, R {s['rec']:.3f}) | "
              f"AUPRC {s['auprc']:.3f} | AUROC {s['auroc']:.3f}")
    return thr, v, t

thr, v, t = report("Kumo Relational, zero-shot", val_p, te_p)

Kumo Relational, zero-shot | threshold 0.982 (chosen on val)
  val  F1 0.4564 (P 0.568, R 0.381) | AUPRC 0.411 | AUROC 0.983
  test F1 0.4572 (P 0.598, R 0.370) | AUPRC 0.425 | AUROC 0.979


## 8. Recall per laundering pattern
Uses the `laundering_type` tags from `HI-Small_Patterns.txt`. The `multi_gin` column is copied from the output of `multignn_pyg.ipynb` (full data, best epoch).

Expect near-zero recall for "(illicit, not in a pattern)". That group contains some ACH laundering and almost all laundering paid by cheque, credit card, cash or Bitcoin (only one such transaction belongs to a pattern). Laundering is far rarer in those formats than in ACH (0.01–0.04% of transactions vs 0.6–1.2%). Only 6% of them have any other laundering transaction in their neighbourhood. The model does rank them above legitimate transactions of the same format, but flagging them isn't worth it: on validation the best possible F1 for any non-ACH format alone is ≤ 0.04, at precision ≤ 2%. So at the F1-maximising threshold, none are flagged.

In [10]:
MULTI_GIN_RECALL = {"(illicit, not in a pattern)": 0.000, "Bipartite": 0.136, "Cycle": 0.141, "Fan-In": 0.252,
                    "Fan-Out": 0.687, "Gather-Scatter": 0.580, "Random": 0.155, "Scatter-Gather": 0.435, "Stack": 0.077}

def pattern_recall(te_p, thr):
    res = txn.iloc[te_idx][["is_laundering", "laundering_type"]].copy()
    res["pred"] = te_p >= thr
    ill = res[res.is_laundering == 1].copy()
    ill["pattern"] = ill.laundering_type.fillna("(illicit, not in a pattern)")
    return ill.groupby("pattern").agg(n=("pred", "size"), recall=("pred", "mean"))

per_pattern = pattern_recall(te_p, thr)
per_pattern["multi_gin"] = per_pattern.index.map(MULTI_GIN_RECALL)
print(per_pattern.round(3))

                               n  recall  multi_gin
pattern                                            
(illicit, not in a pattern)  371   0.013      0.000
Bipartite                     66   0.379      0.136
Cycle                         99   0.333      0.141
Fan-In                       123   0.431      0.252
Fan-Out                      131   0.626      0.687
Gather-Scatter               381   0.514      0.580
Random                        84   0.417      0.155
Scatter-Gather               239   0.502      0.435
Stack                        117   0.402      0.077


## 9. Optional: ablation
Set `RUN_ABLATION = True`. Each row refits the same labeled examples with a different setting and scores smaller evaluation samples (2% / 5% of legitimate transactions), taking about 20–30 minutes per row. The first row is the plain tabular baseline: the model sees only the transaction's own row, with no graph. The last row uses Kumo's default calendar fields.

In [11]:
RUN_ABLATION = False
if RUN_ABLATION:
    KUMO_DEFAULT_CALENDAR = ("minute", "hour", "weekday", "day_of_month", "month")
    variants = {
        "transaction row only (no graph)":       dict(time_mode="strict", num_neighbors=[]),
        "2 hops, strict":                        dict(time_mode="strict", num_neighbors=[16, 16]),
        "2 hops, window":                        dict(time_mode="window", num_neighbors=[16, 16]),
        "3 hops, strict":                        dict(time_mode="strict", num_neighbors=[16, 16, 16]),
        "2 hops, strict, Kumo default calendar": dict(time_mode="strict", num_neighbors=[16, 16],
                                                      calendar_fields=KUMO_DEFAULT_CALENDAR),
    }
    a_rng = np.random.default_rng(SEED + 1)
    a_val, a_val_w = eval_set(n_tr, n_trval, 0.02, a_rng)
    a_te, a_te_w = eval_set(n_trval, n_all, 0.05, a_rng)
    rows = []
    for name, kw in variants.items():
        t0 = time.time()
        fit(model, draws, **kw)
        vp = predict_proba(a_val, kw["time_mode"], kw["num_neighbors"])
        tp_ = predict_proba(a_te, kw["time_mode"], kw["num_neighbors"])
        th = best_threshold(y[a_val], vp, a_val_w)
        sv, st = scores(y[a_val], vp, a_val_w, th), scores(y[a_te], tp_, a_te_w, th)
        rows.append(dict(variant=name, val_f1=sv["f1"], val_auprc=sv["auprc"], test_f1=st["f1"],
                         test_prec=st["prec"], test_rec=st["rec"], test_auprc=st["auprc"]))
        print(f"{name}: test F1 {st['f1']:.3f} ({(time.time()-t0)/60:.1f} min)")
    print(pd.DataFrame(rows).round(3).to_string(index=False))
    fit(model, draws)                              # restore the main configuration

## 10. Optional: fine-tuning
Set `RUN_FINETUNE = True`. This updates the pretrained weights by gradient descent, following NVIDIA's fine-tuning recipe for these models. Each step draws two **disjoint** sets of training transactions: labeled examples, and query transactions whose predicted labels are scored with cross-entropy. Checkpoints are compared by AUPRC on a small validation sample, with the pretrained model as one of the candidates. On CPU a step takes several seconds, so with the defaults below the run takes roughly 45–60 minutes, plus a full re-scoring of validation and test (~1 h).

In [12]:
RUN_FINETUNE = False
FT = dict(epochs=5, steps_per_epoch=20, context_size=1000, context_pos_frac=0.25, query_size=256, query_pos_frac=0.5, lr=1e-5)
ft_result = None
if RUN_FINETUNE:
    train_recipe = make_recipe()
    train_recipe.target = sp.StypeDispatch(                      # as in NVIDIA's fine-tuning example
        categorical=[sp.AlignCategories(), sp.ShuffleCategories(method="shift")], numerical=sp.Standardize())
    train_recipe.output = sp.Identity()                          # raw class scores, no softmax / averaging
    ft_rng = np.random.default_rng(SEED + 2)
    ft_val, ft_val_w = eval_set(n_tr, n_trval, 0.01, ft_rng)    # small sample for checkpoint selection

    def ft_val_auprc():
        model.eval()
        fit(model, draws)
        return average_precision_score(y[ft_val], predict_proba(ft_val), sample_weight=ft_val_w)

    c_pos = min(int(FT["context_size"] * FT["context_pos_frac"]), len(tr_pos) // 2)
    q_pos = min(int(FT["query_size"] * FT["query_pos_frac"]), len(tr_pos) - c_pos)
    c_neg, q_neg = FT["context_size"] - c_pos, FT["query_size"] - q_pos
    opt = torch.optim.AdamW(model.parameters(), lr=FT["lr"])
    best_score, best_state = ft_val_auprc(), copy.deepcopy(model.state_dict())
    print(f"epoch 0 (pretrained) | val AUPRC {best_score:.3f}")
    for epoch in range(1, FT["epochs"] + 1):
        model.train(); losses = []; t0 = time.time()
        for _ in range(FT["steps_per_epoch"]):
            pos = ft_rng.choice(tr_pos, c_pos + q_pos, replace=False)    # disjoint context / query
            neg = ft_rng.choice(tr_neg, c_neg + q_neg, replace=False)
            c, c_rel = sample(task_table(np.sort(np.r_[pos[:c_pos], neg[:c_neg]]))).to(device)
            q, q_rel = sample(task_table(np.sort(np.r_[pos[c_pos:], neg[c_neg:]]))).to(device)
            out = model(x_context=c.drop_columns("label"), y_context=c["label"], x_query=q.drop_columns("label"),
                        related_context_tables=c_rel, related_query_tables=q_rel, recipe=train_recipe)[0]
            logits, target = sdm.evaluation.to_class_indices(out, q["label"], missing_score=-torch.inf)
            loss = F.cross_entropy(logits, target)
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item())
        score = ft_val_auprc()
        print(f"epoch {epoch} | loss {np.mean(losses):.4f} | val AUPRC {score:.3f} | {time.time()-t0:.0f}s")
        if score > best_score:
            best_score, best_state = score, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state); model.eval()
    fit(model, draws)
    val_p_ft, te_p_ft = predict_proba(val_idx, desc="val"), predict_proba(te_idx, desc="test")
    ft_result = report("Kumo Relational, fine-tuned", val_p_ft, te_p_ft)

## 11. Comparison with Multi-GIN
The Multi-GIN numbers are copied from the output of `multignn_pyg.ipynb` (full data, 5 epochs, best validation epoch). That model predicts with argmax after training with a class-weighted loss, while Kumo's threshold is tuned on validation. Multi-GIN's notebook does not compute AUPRC.

In [13]:
MULTI_GIN = dict(val_f1=0.294, test_f1=0.414, test_prec=0.647, test_rec=0.305, test_auprc=np.nan)
sees = {"strict": "only earlier transactions", "window": "its split up to the end"}
rows = [dict(model="Multi-GIN (multignn_pyg.ipynb)", training="supervised, 5 epochs",
             sees="its split up to the end", **MULTI_GIN)]
results = [("Kumo Relational, zero-shot", "none (in-context)", (thr, v, t))]
if ft_result is not None:
    results.append(("Kumo Relational, fine-tuned", f"fine-tuned, {FT['epochs']} epochs", ft_result))
for name, training, (_, sv, st) in results:
    rows.append(dict(model=name, training=training, sees=sees[TIME_MODE], val_f1=sv["f1"], test_f1=st["f1"],
                     test_prec=st["prec"], test_rec=st["rec"], test_auprc=st["auprc"]))
print(pd.DataFrame(rows).round(3).to_string(index=False))
if min(NEG_RATE.values()) < 1:
    print(f"\nKumo precision/F1/AUPRC are estimates from {NEG_RATE['test']:.0%} of the legitimate test transactions "
          f"(all illicit ones scored); set NEG_RATE to 1.0 for exact values.")
if MAX_EDGES:
    print("\nNote: MAX_EDGES is set, so the Kumo numbers are not comparable to the full-data Multi-GIN numbers.")

                         model             training                      sees  val_f1  test_f1  test_prec  test_rec  test_auprc
Multi-GIN (multignn_pyg.ipynb) supervised, 5 epochs   its split up to the end   0.294    0.414      0.647     0.305         NaN
    Kumo Relational, zero-shot    none (in-context) only earlier transactions   0.456    0.457      0.598     0.370       0.425

Kumo precision/F1/AUPRC are estimates from 10% of the legitimate test transactions (all illicit ones scored); set NEG_RATE to 1.0 for exact values.
